In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
import rich
%load_ext rich

In [3]:
import scanpy as sc
import muon as mu
import scorphan as so
import rapids_singlecell as rsc

In [4]:
from pathlib import Path

mod_dir = Path.home() / "eternal_gallery" / "scorphan"
test_dir = mod_dir / "tests"
data_dir = test_dir / "data"

In [5]:
mdata = mu.read(data_dir / "memory_b_cells.h5mu")

In [6]:
# skip now that it has been run once
# rsc.pp.pca(mdata["rna"])
# rsc.pp.pca(mdata["prot"])
# rsc.pp.neighbors(mdata["rna"])
# rsc.pp.neighbors(mdata["prot"])

In [6]:
# mu.pp.neighbors(mdata, key_added="wnn")

In [7]:
# mu.pp.neighbors(mdata, key_added="wnn")
# rsc.tl.leiden(mdata, resolution=1.0, neighbors_key="wnn", key_added="leiden_wnn")
# mu.tl.umap(mdata, neighbors_key="wnn", method="rapids")

[2025-12-04 16:47:03.706] [CUML] [info] build_algo set to brute_force_knn because random_state is given


In [12]:
# mdata.write(data_dir / "memory_b_cells.h5mu")

... storing 'feature_types' as categorical
... storing 'genome' as categorical


In [13]:
from typing import Final
LARGE_NUMBER_OF_OBSERVATIONS: Final[int] = 50000
LOW_MEMORY_SPARSE_SPLITS: Final[int] = 10000
LARGE_MEMORY_SPARSE_SPLITS: Final[int] = 30000

In [14]:
import cupyx

In [15]:
from cupyx.scipy.spatial.distance import cdist

In [16]:
from scanpy.tools._utils import _choose_representation

In [17]:
modality_weights = None
manual_weights = None
n_neighbors = None
n_bandwidth_neighbors = 20
n_multineighbors = 200
neighbor_keys = None
metric = "euclidean"
low_memory = None
key_added = None
weight_key = "mod_weight"
add_weights_to_modalities = False
eps = 1e-4
# method = "rapids"
method="umap"
copy = False
random_state = 42

In [18]:
method

'umap'

In [23]:
if method == "rapids":
    import cupy as np
    import numpy
    # from cupy.random import RandomState
else:
    import numpy as np
    import numpy

In [20]:
neighbor_keys is None

True

In [24]:
randomstate = np.random.RandomState(random_state)
numpy_random_state = numpy.random.RandomState(random_state)
mdata = mdata.copy() if copy else mdata

In [25]:
if neighbor_keys is None:
    modalities = mdata.mod.keys()
    neighbor_keys = {}
else:
    modalities = neighbor_keys.keys()

In [26]:
modalities

dict_keys(['rna', 'prot'])

In [27]:
neighbors_params = {}
reps = {}
observations = mdata.obs.index

In [28]:
if low_memory or (low_memory is None and observations.size > LARGE_NUMBER_OF_OBSERVATIONS):
    sparse_matrix_assign_splits = LOW_MEMORY_SPARSE_SPLITS
else:
    sparse_matrix_assign_splits = LARGE_MEMORY_SPARSE_SPLITS

In [29]:
mod_neighbors = np.empty((len(modalities),), dtype=np.uint16)
mod_reps = {}
mod_n_pcs = {}

In [31]:
for i, mod in enumerate(modalities):
    print(f"{i=} {mod=}")

i=0 mod='rna'
i=1 mod='prot'


In [32]:
neighbor_keys

{}

In [33]:
neighbor_keys.get("rna", "neighbors")

'neighbors'

In [34]:
mdata.mod["rna"].uns["neighbors"]


{
    'connectivities_key': 'connectivities',
    'distances_key': 'distances',
    'params': {'method': 'rapids', 'metric': 'euclidean', 'n_neighbors': 15, 'random_state': 0}
}

In [35]:
for i, mod in enumerate(modalities):
    nkey = neighbor_keys.get(mod, "neighbors")
    try:
        nparams = mdata.mod[mod].uns[nkey]
    except KeyError as err:
        msg = f'Did not find .uns["{nkey}"] for modality "{mod}". Run `sc.pp.neighbors` on all modalities first.'
        raise ValueError(msg) from err

    use_rep = nparams["params"].get("use_rep", None)
    n_pcs = nparams["params"].get("n_pcs", None)
    mod_neighbors[i] = nparams["params"].get("n_neighbors", 0)

    neighbors_params[mod] = nparams
    reps[mod] = _choose_representation(adata=mdata.mod[mod], use_rep=use_rep, n_pcs=n_pcs)
    mod_reps[mod] = use_rep if use_rep is not None else -1  # otherwise this is not saved to h5mu
    mod_n_pcs[mod] = n_pcs if n_pcs is not None else -1

In [36]:
n_neighbors = 15

In [37]:
if n_neighbors is None:
    mod_neighbors = mod_neighbors[mod_neighbors > 0]
    n_neighbors = int(np.round_(np.mean(mod_neighbors), 0))

ratios = np.full((len(observations), len(modalities)), -np.inf, dtype=np.float64)
sigmas = {}

In [38]:
modalities

dict_keys(['rna', 'prot'])

In [39]:
i1 = 0
mod1 = "rna"

here begins the loop

In [88]:
type(observations.isin(observations1))

<class 'numpy.ndarray'>

In [41]:
# for i1, mod1 in enumerate(modalities):
observations1 = observations.intersection(mdata.mod[mod1].obs.index)
ratioidx = np.where(np.array(observations.isin(observations1)))[0] # recast to an array so that if we're using cupy, it is converted to a cupy.array
nparams1 = neighbors_params[mod1]
X = reps[mod1]  # noqa: N806
neighbordistances = mdata.mod[mod1].obsp[nparams1["distances_key"]]
nndistances = np.empty((neighbordistances.shape[0],), neighbordistances.dtype)
# neighborsdistances is a sparse matrix, we can either convert to dense, or loop
for i in range(neighbordistances.shape[0]):
    nndist = neighbordistances[i, :].data
    if nndist.size == 0:
        msg = (
            f"Cell {i} in modality {mod1} does not have any neighbors. "
            "This could be due to subsetting after nearest neighbors calculation. "
            "Make sure to subset before calculating nearest neighbors."
        )
        raise ValueError(msg)
    nndistances[i] = nndist.min()

In [42]:
from scipy.sparse import issparse

In [43]:
issparse(X)

False

In [44]:
type(np.array(X))

<class 'numpy.ndarray'>

In [45]:
import numpy

In [46]:
numpy.ptp(numpy.array(X), axis=0)


array([13.577059 , 10.659903 , 14.273691 ,  7.8594074,  8.518041 ,
        6.123703 ,  7.78772  ,  7.6495304,  6.307673 ,  7.1014953,
        7.416692 ,  6.127624 ,  6.45592  ,  6.288514 ,  5.918789 ,
        5.7568545,  5.1813455,  5.381506 ,  5.2957363,  5.7675133,
        6.0494366,  5.684757 ,  6.3680735,  4.295512 ,  4.991604 ,
        5.4326706,  5.854253 ,  5.9527197,  4.8174486,  4.702623 ,
        5.447224 ,  4.568671 ,  5.701375 ,  5.8237467,  4.45933  ,
        4.9849486,  4.3812275,  5.2885275,  5.7491345,  4.6416245,
        4.5730495,  4.4697647,  4.9130874,  5.202205 ,  4.2542977,
        4.95506  ,  4.756651 ,  4.47804  ,  4.1011395,  4.2503405],
      dtype=float32)

In [47]:
np.ptp(np.array(X), axis=0)


array([13.577059 , 10.659903 , 14.273691 ,  7.8594074,  8.518041 ,
        6.123703 ,  7.78772  ,  7.6495304,  6.307673 ,  7.1014953,
        7.416692 ,  6.127624 ,  6.45592  ,  6.288514 ,  5.918789 ,
        5.7568545,  5.1813455,  5.381506 ,  5.2957363,  5.7675133,
        6.0494366,  5.684757 ,  6.3680735,  4.295512 ,  4.991604 ,
        5.4326706,  5.854253 ,  5.9527197,  4.8174486,  4.702623 ,
        5.447224 ,  4.568671 ,  5.701375 ,  5.8237467,  4.45933  ,
        4.9849486,  4.3812275,  5.2885275,  5.7491345,  4.6416245,
        4.5730495,  4.4697647,  4.9130874,  5.202205 ,  4.2542977,
        4.95506  ,  4.756651 ,  4.47804  ,  4.1011395,  4.2503405],
      dtype=float32)

In [48]:
# We want to get the k-nn with the largest Jaccard distance, but break ties using
# Euclidean distance. Largest Jaccard distance corresponds to lowest Jaccard index,
# i.e. 1 - Jaccard distance. The naive version would be to compute pairwise Jaccard and
# Euclidean distances for all points, but this is low and needs lots of memory. We
# want to use an efficient k-nn algorithm, however no package that I know of supports
# tie-breaking k-nn, so we use a custom distance. Jaccard index is always between 0 and 1
# and has discrete increments of at least 1/N, where N is the number of data points.
# If we scale the Jaccard indices by N, the minimum Jaccard index will be 1. If we scale
# all Euclidean distances to be less than one, we can define a combined distance as the
# sum of the scaled Jaccard index and one minus the Euclidean distances. This is not a
# proper metric, but UMAP's nearest neighbor search uses NN-descent, which works with
# arbitrary similarity measures.
# The scaling factor for the Euclidean distance is given by the length of the diagonal
# of the bounding box of the data. This can be computed in linear time by just taking
# the minimal and maximal coordinates of each dimension.
num_obs = X.shape[0]

# recast X here because if we are using cupy, we need this to be a cupy.array as a numpy.array will
# fail due to numpy.array.ptp being removed and if we are using numpy and it was already a numpy.array,
# recasting has no effect
bbox_norm = np.linalg.norm(_sparse_csr_ptp(X) if issparse(X) else np.ptp(np.array(X), axis=0), ord=2)
lmemory = low_memory if low_memory is not None else num_obs > LARGE_NUMBER_OF_OBSERVATIONS

In [49]:
from numba import njit

In [62]:
from umap.distances import euclidean
from umap.sparse import sparse_euclidean, sparse_jaccard
from umap.umap_ import nearest_neighbors

In [63]:
_euclidean = njit(euclidean.py_func, inline="always", fastmath=True)
_sparse_euclidean = njit(sparse_euclidean.py_func, inline="always")
_sparse_jaccard = njit(sparse_jaccard.py_func, inline="always")

In [64]:
@njit
def _jaccard_euclidean_metric(
    x: int,
    y: int,
    X: np.ndarray,
    neighbors_indices: np.ndarray,
    neighbors_indptr: np.ndarray,
    neighbors_data: np.ndarray,
    N: int,
    bbox_norm: float,
):
    x = int(x[0])  # this is for compatibility with pynndescent
    y = int(y[0])  # pynndescent converts the data to float32
    if x == y:
        return N + 1.0

    from_inds = neighbors_indices[neighbors_indptr[x] : neighbors_indptr[x + 1]]
    from_data = neighbors_data[neighbors_indptr[x] : neighbors_indptr[x + 1]]
    to_inds = neighbors_indices[neighbors_indptr[y] : neighbors_indptr[y + 1]]
    to_data = neighbors_data[neighbors_indptr[y] : neighbors_indptr[y + 1]]
    jac = _sparse_jaccard(from_inds, from_data, to_inds, to_data)

    if jac < 1.0:
        return (N - jac * N) + (bbox_norm - _euclidean(X[x, :], X[y, :])) / bbox_norm
    else:
        return N + 1.0


In [65]:
if issparse(X):
    X = X.tocsr()  # noqa: N806
    cmetric = _jaccard_sparse_euclidean_metric
    metric_kwds = {
        "X_indices": X.indices,
        "X_indptr": X.indptr,
        "X_data": X.data,
        "neighbors_indices": neighbordistances.indices,
        "neighbors_indptr": neighbordistances.indptr,
        "neighbors_data": neighbordistances.data,
        "N": num_obs,
        "bbox_norm": bbox_norm,
    }
else:
    cmetric = _jaccard_euclidean_metric
    metric_kwds = {
        "X": X,
        "neighbors_indices": neighbordistances.indices,
        "neighbors_indptr": neighbordistances.indptr,
        "neighbors_data": neighbordistances.data,
        "N": num_obs,
        "bbox_norm": bbox_norm,
    }

In [52]:
import logging

In [53]:
logging.info(f"Calculating kernel bandwidth for '{mod1}' modality...")

2025-12-04 16:48:55 | [INFO] Calculating kernel bandwidth for 'rna' modality...


In [55]:
import importlib.metadata

In [56]:
importlib.metadata.version("pynndescent")

'0.5.13'

In [66]:
if method == "rapids":
    from cuml.neighbors import NearestNeighbors

    x = np.arange(num_obs)[:, np.newaxis]
    nn = NearestNeighbors(
        n_neighbors=n_bandwidth_neighbors,
        algorithm="brute",
        metric=metric,
        output_type="cupy",
        metric_params=metric_kwds,
    )
    nn.fit(x)
    _, nn_indices = nn.kneighbors(
        X=x,
        n_neighbors=n_bandwidth_neighbors,
    )
    nn_indices = nn_indices.get()
elif method == "umap":
    from umap.umap_ import nearest_neighbors

    nn_indices, _, _ = nearest_neighbors(
        X=np.arange(num_obs)[:, np.newaxis],
        n_neighbors=n_bandwidth_neighbors,
        metric=cmetric,
        metric_kwds=metric_kwds,
        random_state=randomstate,
        angular=False,
        low_memory=lmemory,
    )

In [149]:
from pynndescent import NNDescent

In [ ]:
NNDescent(

In [67]:
metric_kwds


{
    'X': array([[ 1.0041928 ,  0.48779592,  0.9183103 , ...,  0.0054481 ,
         0.51966774, -0.5951595 ],
       [ 1.6160522 ,  0.00997004,  0.37271702, ..., -1.6473706 ,
         0.8215291 ,  0.18958999],
       [ 4.2911835 ,  0.35535026,  2.295598  , ..., -0.34964576,
        -0.5779942 , -0.9544124 ],
       ...,
       [-1.6906195 ,  1.6206521 ,  0.31098545, ...,  0.521443  ,
        -0.6118361 ,  0.02563541],
       [ 2.4452844 ,  2.0095315 , -0.41895986, ...,  0.04783937,
        -0.13362074,  0.5912153 ],
       [ 0.92055917,  2.691083  ,  1.0872908 , ...,  0.15140763,
        -0.07737958,  0.27753884]], shape=(12996, 50), dtype=float32),
    'neighbors_indices': array([    0,    30,    14, ...,  8563, 10878, 12839],
      shape=(194940,), dtype=int32),
    'neighbors_indptr': array([     0,     15,     30, ..., 194910, 194925, 194940],
      shape=(12997,), dtype=int32),
    'neighbors_data': array([0.       , 3.3263552, 3.3790767, ..., 4.15408  , 4.1905055,
       4.1974

In [58]:
from umap.umap_ import nearest_neighbors

In [69]:
nn_indices, _, _ = nearest_neighbors(
    X=np.arange(num_obs)[:, np.newaxis],
    n_neighbors=n_bandwidth_neighbors,
    metric=cmetric,
    metric_kwds=metric_kwds,
    random_state=random_state,
    angular=False,
    low_memory=lmemory,
)

In [70]:
nn_indices


array([[   36,    45,    14, ...,     5,     8,     3],
       [   63,     9,    56, ...,     4,     6,     2],
       [ 3948,  3719,  8527, ...,    35,  8448,  8516],
       ...,
       [12781, 12925, 12879, ..., 12990, 12890, 12955],
       [12854, 12922, 12925, ..., 12802, 12924, 12904],
       [12854, 12932, 12887, ..., 12994, 12840, 12949]],
      shape=(12996, 20), dtype=int32)

In [76]:
neighbordistances.data


array([0.       , 3.3263552, 3.3790767, ..., 4.15408  , 4.1905055,
       4.19747  ], shape=(194940,), dtype=float32)

In [71]:
import cupy as cp

In [82]:
# bbox_norm = np.linalg.norm(_sparse_csr_ptp(X) if issparse(X) else np.ptp(np.array(X), axis=0), ord=2)
cp.linalg.norm(_sparse_csr_ptp(X) if issparse(X) else cp.ptp(cp.array(X), axis=0), ord=2)

array(44.89023, dtype=float32)

In [83]:
cp_metric_kwds = {
    "X": cp.array(X),
    "neighbors_indices": cp.array(neighbordistances.indices),
    "neighbors_indptr": cp.array(neighbordistances.indptr),
    "neighbors_data": cp.array(neighbordistances.data),
    "N": num_obs,
    "bbox_norm": cp.linalg.norm(_sparse_csr_ptp(X) if issparse(X) else cp.ptp(cp.array(X), axis=0), ord=2),
}

In [84]:
from cuml.neighbors import NearestNeighbors

In [88]:
cp_x = cp.arange(num_obs)[:, cp.newaxis]

In [97]:
cp_nn = NearestNeighbors(
    n_neighbors=n_bandwidth_neighbors,
    algorithm="brute",
    metric=metric,
    output_type="cupy",
    metric_params=cp_metric_kwds,
)

In [101]:
cp_nn

NearestNeighbors()

In [153]:
from cuvs.neighbors import nn_descent, brute_force

In [150]:
build_params = nn_descent.IndexParams("euclidean", metric_kwds, intermediate_graph_degree=20, graph_degree=20)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 build_params = nn_descent.IndexParams("euclidean", metric_kwds, intermediate_graph_degre     │
│   2                                                                                              │
│                                                                                                  │
│ in cuvs.neighbors.nn_descent.nn_descent.IndexParams.__init__:88                                  │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: __init__() takes exactly 0 positional arguments (2 given)

In [147]:
index = nn_descent.build(build_params, cp.array(X))

In [148]:
index.graph


array([[   30,    14,  6369, ...,  6711,  6434,  3692],
       [ 5730,  3983,  5585, ...,  3731,  7577, 11223],
       [ 2239,  5356,  1420, ...,  5146,  1293,  8653],
       ...,
       [12930, 12965, 10651, ..., 10528, 12837, 12841],
       [12847, 10672, 10394, ..., 12966, 10717, 10802],
       [ 6579, 12979, 10715, ..., 12847, 10782, 12814]],
      shape=(12996, 20), dtype=uint32)

In [156]:
metric_kwds["X"]


array([[ 1.0041928 ,  0.48779592,  0.9183103 , ...,  0.0054481 ,
         0.51966774, -0.5951595 ],
       [ 1.6160522 ,  0.00997004,  0.37271702, ..., -1.6473706 ,
         0.8215291 ,  0.18958999],
       [ 4.2911835 ,  0.35535026,  2.295598  , ..., -0.34964576,
        -0.5779942 , -0.9544124 ],
       ...,
       [-1.6906195 ,  1.6206521 ,  0.31098545, ...,  0.521443  ,
        -0.6118361 ,  0.02563541],
       [ 2.4452844 ,  2.0095315 , -0.41895986, ...,  0.04783937,
        -0.13362074,  0.5912153 ],
       [ 0.92055917,  2.691083  ,  1.0872908 , ...,  0.15140763,
        -0.07737958,  0.27753884]], shape=(12996, 50), dtype=float32)

In [162]:
cp_x.astype(float)


array([[0.0000e+00],
       [1.0000e+00],
       [2.0000e+00],
       ...,
       [1.2993e+04],
       [1.2994e+04],
       [1.2995e+04]], shape=(12996, 1))

In [166]:
brute_build_params = brute_force.build(cp.array(X), metric="euclidean")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 brute_build_params = brute_force.build(cp.array(X), metric="euclidean")                      │
│   2                                                                                              │
│                                                                                                  │
│ in cuvs.common.resources.auto_sync_resources.wrapper:110                                         │
│                                                                                                  │
│ in cuvs.neighbors.brute_force.brute_force.build:116                                              │
│                                                                                                  │
│ in cuvs.neighbors.brute_force.brute_force.build:117                                              │
│                                                                                                  │
│ in cuvs.common.exceptions.check_cuvs:37                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
CuvsException: CUDA error encountered at: 
file=/pyenv/versions/3.13.7/lib/python3.13/site-packages/libraft/include/raft/linalg/detail/coalesced_reduction-inl
.cuh line=271: call='cudaPeekAtLastError()', Reason=cudaErrorInvalidValue:invalid argument
Obtained 38 stack frames
#1 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs.so: 
raft::cuda_error::cuda_error(std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> > 
const&) +0x9d [0x70b4f4eaa2ad]
#2 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs.so(+0x3f0151) 
[0x70b4f4d4f151]
#3 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs.so(+0xe828f3) 
[0x70b4f57e18f3]
#4 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs.so: 
cuvs::neighbors::brute_force::index<float, float> cuvs::neighbors::detail::build<float, float, 
raft::host_device_accessor<std::experimental::default_accessor<float const>, (raft::memory_type)2>, 
std::experimental::layout_right>(raft::resources const&, std::experimental::mdspan<float const, 
std::experimental::extents<long, 18446744073709551615ul, 18446744073709551615ul>, std::experimental::layout_right, 
raft::host_device_accessor<std::experimental::default_accessor<float const>, (raft::memory_type)2> >, 
cuvsDistanceType, float) +0x288 [0x70b4f57f9018]
#5 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs.so: 
cuvs::neighbors::brute_force::build(raft::resources const&, cuvs::neighbors::brute_force::index_params const&, 
std::experimental::mdspan<float const, std::experimental::extents<long, 18446744073709551615ul, 
18446744073709551615ul>, std::experimental::layout_right, 
raft::host_device_accessor<std::experimental::default_accessor<float const>, (raft::memory_type)2> >) +0x28 
[0x70b4f57e4178]
#6 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs_c.so(+0x6daef) 
[0x70b4f4812aef]
#7 in /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/libcuvs/lib64/libcuvs_c.so: 
cuvsBruteForceBuild +0x27 [0x70b4f4812f77]
#8 in 
/mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/cuvs/neighbors/brute_force/brute_force.cpython-3
13-x86_64-linux-gnu.so(+0x11ea2) [0x70b42004bea2]
#9 in 
/mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/cuvs/neighbors/brute_force/brute_force.cpython-3
13-x86_64-linux-gnu.so(+0x13ebc) [0x70b42004debc]
#10 in 
/mnt/d/eternal_gallery/scorphan/.

In [117]:
cp_nn.fit(cp_x)
cp_distances, cp_nn_indices = cp_nn.kneighbor_graph(
    X=,
    n_neighbors=3,
)
cp_nn_indices = cp_nn_indices.get()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 cp_nn.fit(cp_x)                                                                              │
│ ❱ 2 cp_distances, cp_nn_indices = cp_nn.kneighbors(                                              │
│   3 │   X=cp.array(X),                                                                           │
│   4 │   n_neighbors=3,                                                                           │
│   5 )                                                                                            │
│                                                                                                  │
│ /mnt/d/eternal_gallery/scorphan/.venv/lib/python3.13/site-packages/cuml/internals/api_decorators │
│ .py:211 in wrapper                                                                               │
│                                                                                                  │
│   208 │   │   │   │   │   │   set_api_output_dtype(output_dtype)                                 │
│   209 │   │   │   │   │                                                                          │
│   210 │   │   │   │   │   if process_return:                                                     │
│ ❱ 211 │   │   │   │   │   │   ret = func(*args, **kwargs)                                        │
│   212 │   │   │   │   │   else:                                                                  │
│   213 │   │   │   │   │   │   return func(*args, **kwargs)                                       │
│   214                                                                                            │
│                                                                                                  │
│ in cuml.neighbors.nearest_neighbors.NearestNeighbors.kneighbors:768                              │
│                                                                                                  │
│ in cuml.neighbors.nearest_neighbors.NearestNeighbors._kneighbors_internal:848                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValueError: Dimensions of X need to match dimensions of indices (1)

In [113]:
np.arange(num_obs)[:, np.newaxis]


array([[    0],
       [    1],
       [    2],
       ...,
       [12993],
       [12994],
       [12995]], shape=(12996, 1))

In [108]:
cp_nn_indices


array([[    0,     1,     2],
       [    1,     0,     2],
       [    2,     1,     3],
       ...,
       [12992, 12993, 12994],
       [12992, 12993, 12994],
       [12992, 12993, 12994]], shape=(12996, 3))

In [106]:
cp_nn.kneighbors_graph(X=cp_x, n_neighbors=20).toarray()


array([[1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       [1., 1., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 1., 1., 1.],
       [0., 0., 0., ..., 1., 1., 1.]], shape=(12996, 12996), dtype=float32)

In [99]:
cp_nn_indices


array([[    0,     1,     2, ...,    17,    18,    19],
       [    1,     0,     2, ...,    17,    18,    19],
       [    2,     1,     3, ...,    17,    18,    19],
       ...,
       [12992, 12993, 12994, ..., 12979, 12977, 12976],
       [12992, 12993, 12994, ..., 12979, 12977, 12976],
       [12992, 12993, 12994, ..., 12978, 12976, 12977]], shape=(12996, 20))

In [100]:
other_stuff


array([[ 0.      ,  1.      ,  2.      , ..., 17.      , 18.      ,
        19.      ],
       [ 0.      ,  1.      ,  1.      , ..., 16.      , 17.      ,
        18.      ],
       [ 0.      ,  1.      ,  1.      , ..., 15.      , 16.      ,
        17.      ],
       ...,
       [ 0.      ,  0.      ,  0.      , ..., 14.96663 , 16.      ,
        16.970562],
       [ 0.      ,  0.      ,  0.      , ..., 16.      , 16.970562,
        17.888544],
       [ 0.      ,  0.      ,  0.      , ..., 16.970562, 18.761663,
        18.761663]], shape=(12996, 20), dtype=float32)

In [ ]:
csigmas = np.empty((num_obs,), dtype=neighbordistances.dtype)

In [ ]:
if issparse(X):
    for i, neighbors in enumerate(nn_indices):
        csigmas[i] = cdist(X[i : (i + 1), :].toarray(), X[neighbors, :].toarray(), metric="euclidean").mean()
else:
    for i, neighbors in enumerate(nn_indices):
        csigmas[i] = cdist(X[i : (i + 1), :], X[neighbors, :], metric="euclidean").mean()

In [ ]:
if not manual_weights:
    currtheta = None
    thetas = np.full((len(observations1), len(modalities) - 1), -np.inf, dtype=neighbordistances.dtype)

    lasti = 0

    logging.info(f"Calculating cell affinities for '{mod1} modality...")
    for i2, mod2 in enumerate(modalities):
        nparams2 = neighbors_params[mod2]
        neighbordistances = mdata.mod[mod2].obsp[nparams2["distances_key"]]
        observations2 = observations1.intersection(mdata.mod[mod2].obs.index)
        Xidx = np.where(observations1.isin(observations2))[0]  # noqa: N806
        r = np.empty(shape=(len(observations2), X.shape[1]), dtype=neighbordistances.dtype)
        # alternative to the loop would be broadcasting, but this requires converting the sparse
        # connectivity matrix to a dense ndarray and producing a temporary 3d array of size
        # n_cells x n_cells x n_genes => requires a lot of memory
        for i, cell in enumerate(Xidx):
            r[i, :] = np.asarray(np.mean(X[neighbordistances[cell, :].nonzero()[1], :], axis=0)).squeeze()

        theta = np.exp(
            -np.maximum(np.linalg.norm(X[Xidx, :] - r, ord=2, axis=-1) - nndistances[Xidx], 0)
            / (csigmas[Xidx] - nndistances[Xidx])
        )
        if i1 == i2:
            currtheta = theta
        else:
            thetas[:, lasti] = theta
            lasti += 1
    ratios[ratioidx, i1] = currtheta / (np.max(thetas, axis=1) + eps)
sigmas[mod1] = csigmas

here ends the loop

In [ ]:
if manual_weights:
    for mod in modalities:
        if mod not in manual_weights:
            msg = f"A weight was not suppied for the {mod} modality"
            raise ValueError(msg)
    weights = np.empty((len(mdata.obs), len(modalities)))
    for i, mod in enumerate(manual_weights):
        if isinstance(manual_weights[mod], numbers.Number):
            manual_weights[mod] = np.broadcast_to(manual_weights[mod], len(mdata[mod].obs))
        if len(manual_weights[mod]) != len(mdata[mod].obs):
            msg = f"The length of the weight array for {mod} does not match the number of cells. Either suppy a single value or one for each cell"
            raise ValueError(msg)
        weights[:, i] = manual_weights[mod]
else:
    weights = softmax(ratios, axis=1)

In [ ]:
neighbordistances = csr_matrix((mdata.n_obs, mdata.n_obs), dtype=np.float64)
largeidx = mdata.n_obs**2 > np.iinfo(np.int32).max
if largeidx:  # work around scipy bug https://github.com/scipy/scipy/issues/13155
    neighbordistances.indptr = neighbordistances.indptr.astype(np.int64)
    neighbordistances.indices = neighbordistances.indices.astype(np.int64)
for _, m in enumerate(modalities):
    cmetric = neighbors_params[m].get("metric", "euclidean")
    observations1 = observations.intersection(mdata.mod[m].obs.index)

    rep = reps[m]
    lmemory = low_memory if low_memory is not None else rep.shape[0] > LARGE_NUMBER_OF_OBSERVATIONS
    logger.info(f"Calculating nearest neighbor candidates for '{m}' modality...")
    logger.debug(f"Using low_memory={lmemory} for '{m}' modality")

    if method == "rapids":
        nn = NearestNeighbors(
                n_neighbors=n_bandwidth_neighbors,
                algorithm="brute",
                metric=metric,
                output_type="cupy",
                metric_params=metric_kwds,
            )
        nn.fit(x)
        distances, nn_indices = nn.kneighbors(
            X=x,
            n_neighbors=n_bandwidth_neighbors,
        )
        nn_indices = nn_indices.get()
        distances = distances.get()
    elif method == "umap":
        nn_indices, distances, _ = nearest_neighbors(
            rep,
            n_neighbors=n_multineighbors + 1,
            metric=cmetric,
            metric_kwds={},
            random_state=randomstate,
            angular=False,
            low_memory=lmemory,
        )

    graph = csr_matrix(
        (
            distances[:, 1:].reshape(-1),
            nn_indices[:, 1:].reshape(-1),
            np.concatenate((nn_indices[:, 0] * n_multineighbors, (nn_indices[:, 1:].size,))),
        ),
        shape=(rep.shape[0], rep.shape[0]),
    )
    with warnings.catch_warnings():
        # CSR is faster here than LIL, no matter what SciPy says
        warnings.simplefilter("ignore", category=SparseEfficiencyWarning)
        if observations1.size == observations.size:
            if neighbordistances.size == 0:
                neighbordistances = graph
            else:
                neighbordistances += graph
        # the naive version of neighbordistances[idx[:, np.newaxis], idx[np.newaxis, :]] += graph
        else:
            # uses way too much memory
            if largeidx:
                graph.indptr = graph.indptr.astype(np.int64)
                graph.indices = graph.indices.astype(np.int64)
            fullstarts, fullstops = _make_slice_intervals(
                np.where(observations.isin(observations1))[0], sparse_matrix_assign_splits
            )
            modstarts, modstops = _make_slice_intervals(
                np.where(mdata.mod[m].obs.index.isin(observations1))[0],
                sparse_matrix_assign_splits,
            )

            for fullidxstart1, fullidxstop1, modidxstart1, modidxstop1 in zip(
                fullstarts, fullstops, modstarts, modstops, strict=False
            ):
                for fullidxstart2, fullidxstop2, modidxstart2, modidxstop2 in zip(
                    fullstarts, fullstops, modstarts, modstops, strict=False
                ):
                    neighbordistances[fullidxstart1:fullidxstop1, fullidxstart2:fullidxstop2] += graph[
                        modidxstart1:modidxstop1, modidxstart2:modidxstop2
                    ]

In [ ]:
neighbordistances.data[:] = 0
logging.info("Calculating multimodal nearest neighbors...")
if modality_weights is None:
    modality_weights = {_: 1 for _ in modalities}
if len(modality_weights) != len(modalities):
    msg = "Number of weights in modality_weights does not match the actual number of modalities!"
    raise ValueError(msg)
for i, m in enumerate(modalities):
    observations1 = observations.intersection(mdata.mod[m].obs.index)
    fullidx = np.where(observations.isin(observations1))[0]

    if weight_key:
        if add_weights_to_modalities:
            mdata.mod[m].obs[weight_key] = weights[fullidx, i] * modality_weights[m]
        else:
            # mod_weight -> mod:mod_weight
            mdata.obs[":".join([m, weight_key])] = weights[fullidx, i] * modality_weights[m]

    rep = reps[m]
    csigmas = sigmas[m]

    for cell, _ in enumerate(fullidx):
        row = slice(neighbordistances.indptr[cell], neighbordistances.indptr[cell + 1])
        nz = neighbordistances.indices[row]
        neighbordistances.data[row] += (
            np.exp(neighdist(rep, cell, nz, metric) / csigmas[cell]).squeeze()
            * weights[cell, i]
            * modality_weights[m]
        )
neighbordistances.data = np.sqrt(0.5 * (1 - neighbordistances.data))

In [ ]:
neighbordistances = _sparse_csr_fast_knn(neighbordistances, n_neighbors + 1)

In [ ]:
logging.info("Calculating connectivities...")

if method == "rapids":
    try:
        from cuml.manifold.simpl_set import fuzzy_simplicial_set
    except ImportError as err:
        msg = "cuml could not be imported - is it installed?"
        raise ImportError(msg) from err
elif method == "umap":
    from umap.umap_ import fuzzy_simplicial_set

In [ ]:
connectivities, _, _ = fuzzy_simplicial_set(
    knn_indices=neighbordistances.indices.reshape((neighbordistances.shape[0], n_neighbors + 1)),
    knn_dists=neighbordistances.data.reshape((neighbordistances.shape[0], n_neighbors + 1)),
    n_obs=neighbordistances.shape[0],
    n_neighbors=n_neighbors + 1,
)

In [ ]:
if key_added is None:
    key_added = "neighbors"
    conns_key = "connectivities"
    dists_key = "distances"
else:
    conns_key = key_added + "_connectivities"
    dists_key = key_added + "_distances"
neighbors_dict = {"connectivities_key": conns_key, "distances_key": dists_key}
neighbors_dict["params"] = {
    "n_neighbors": n_neighbors,
    "n_multineighbors": n_multineighbors,
    "metric": metric,
    "eps": eps,
    "random_state": random_state,
    "use_rep": mod_reps,
    "n_pcs": mod_n_pcs,
    "method": method,
}

In [ ]:
mdata.obsp[dists_key] = neighbordistances
mdata.obsp[conns_key] = connectivities
mdata.uns[key_added] = neighbors_dict

mdata.update_obs()

In [3]:
import scanpy as sc

In [5]:
adata = sc.read("/home/milo/eternal_gallery/ana_multiome/data/20250815_combined_rnaseq_asapseq_adt_modalities.h5ad")

In [6]:
adata

AnnData object with n_obs × n_vars = 801520 × 37
    obs: '_scvi_batch', '_scvi_labels', 'leiden', 'source'
    var: 'mean', 'std'
    uns: 'log1p', 'pca'
    obsm: 'X_pca', 'X_pca_harmony', 'X_scAR', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'denoised', 'denoised_normalized', 'raw_counts'

In [14]:
import pandas as pd

In [17]:
pd.DataFrame(adata.obs.value_counts(["source","leiden"]) / adata.obs.shape[0]).sort_index()

count
source leiden          
ASAP   0       0.014300
       1       0.023389
       2       0.016073
       3       0.010072
       4       0.005496
       5       0.003218
       6       0.000957
       7       0.001087
       8       0.024286
       9       0.038220
       10      0.026913
       11      0.024921
       12      0.004927
       13      0.000927
       14      0.000131
RNA    0       0.020626
       1       0.018929
       2       0.011176
       3       0.081793
       4       0.011881
       5       0.031025
       6       0.034502
       7       0.093592
       8       0.008927
       9       0.030418
       10      0.100246
       11      0.071394
       12      0.021235
       13      0.083142
       14      0.056373
       15      0.040914
       16      0.001488
       17      0.023090
       18      0.003606
       19      0.010705
       20      0.004278
       21      0.037176
       22      0.008567

In [8]:
(
    adata
        .obs
        .groupby([
            'source',
            'leiden'
        ])
        .count()
).iloc[:,[0]][
            adata
                .obs
                .groupby([
                    'source',
                    'leiden'
                ])
                .count()
                .iloc[:,0]
                .apply(lambda x: x != 0)
    ]

/tmp/ipykernel_301263/3945670714.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([
/tmp/ipykernel_301263/3945670714.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([


_scvi_batch
source leiden             
ASAP   0             11462
       1             18747
       2             12883
       3              8073
       4              4405
       5              2579
       6               767
       7               871
       8             19466
       9             30634
       10            21571
       11            19975
       12             3949
       13              743
       14              105
RNA    0             16532
       1             15172
       2              8958
       3             65559
       4              9523
       5             24867
       6             27654
       7             75016
       8              7155
       9             24381
       10            80349
       11            57224
       12            17020
       13            66640
       14            45184
       15            32793
       16             1193
       17            18507
       18             2890
       19             8580
       20             3429
       21            29797
       22             6867